# Task A: Mirror for Algernon
Researchers studied whether the presence of a mirror is associated with laboratory mice's choice of where to spend their time.

16 laboratory mice were individually placed in cages consisting of two rooms connected by a tunnel. One of the rooms contained a mirror, while the other did not. During the experiment, the position of the mirror was changed to reduce the influence of room location on the results.

For each mouse, the proportion of time spent in the room with the mirror was determined. The distribution of the resulting values is shown in the histogram.

![Mice-Mirror experiments resulst](../figs/task_A_mice_histogram.png)

The horizontal axis shows the proportion of time spent in the room containing mirror, expressed as a percentage; the vertical axis shows the number of mice.

It is known that each mouse spent more time in one of the two rooms than in the other. Assume that the results for different mice are independent and that the probability of spending more time in the room without a mirror is the same for all mice.

Can we conclude at the `5%` significance level that mice tend to spend more time in one of the two rooms - with or without a mirror?

Propose a statistical test based on the available data, calculate the exact two-sided `p-value`, and draw a conclusion at the significance level $\alpha=0.05$. When calculating the `p-value`, account for deviations from the expected result in both the greater and the lesser directions.

## output format
In the answer field, enter two numbers separated by a space:
* the `p-value` of the statistical test;
* the number `1` if the null hypothesis is rejected at the significance level $\alpha=0.05$, or the number `0` otherwise.

The absolute error of the `p-value` must not exceed $10^{-4}$.

# Solution follows

## some conclusions from the task posing:
1) A mouse can be simulated as sampling of random variable taking binary values when monitored at some moment of time - say `1` if mouse it found in the room with mirror and `0` if otherwise, $p$ - traditionally stands for probability of `1`
2) In fact we also suppose that there are no autocorrelation between subsequent monitoring of the same mouse (for example, the minimal time between two subsequent monitorings is sufficient for a mouse to change the room of its location many times)
3) The probability distribution for all mice monitoring acts is the same, and $H_o$-hypothesis suggests $p=0.5$, yet the sampling instances are different, that's why the mean function taken over each sampling (and represented on the figure) differs among mice
4) What is yet unknown - it is the duration of experiment (or the length of sampling monitoring results during experiment), but we suppose it is the same for all mice.  We also can say that this sampling is short enough to allow for such variation in the results among mices
5) We do not know how hystogram bins include left and right values, thus we shall limit sampling lengthes by odd values only - otherwise it is not clear to which bin should we add the value 0.5 - to the left one or to the right one? 

## brilliant idea
* actually, an MC-simulation of this problem needs heavy modelling, as it is two-steps: for different sampling length that it turn would have different likelyhood at a given histogram.  In addition, we have only fits into histogram bins, but not exact values of mean function, so it means another dimention for uncertainty to be modelled
* but as long as we still need suggestion 5, we can consider more generic random variable whose $H_o$ would match the $H_o$ of the previously discussed "Bernulli mouse" with one monitoring result
* although the "natural" choice for complimentary randon variable for Bernulli is Binomial, it is not the best choice for this case as the length of sampling is not known
* let's consider another binary random variable $X$, that takes values:
  * `1` - if the mean function of one mouse monitoring experiment is greater than 0.5
  * `0` - if the mean function of one mouse monitoring experiment is less than 0.5
  * we consider only odd sampling length, thus the exact value 0.5 is not possible
* in this case $H_o(p=0.5)$ for Bernully mouse is valid only if $E(X)=0.5$, it is not also evidently sufficient, but the logic of checking hypothesis is perfectly fitting this asymmetry: we ether reject $H_o$ with a certain `p-value` or cannot reach the trust limit $\alpha$ on a given sampling to reject it - yet we can never confirm it.  Therefore if we reject $E(X)=0.5$, we also reject $H_o(p=0.5)$ - that's enough for us!

## reposed problem
1) thus we have 16 independent samples of binary random variable $X$ with 13 zeros and 3 ones
2) or it is one sample of binomial random variable $Binomial(n=16,k=X_i)$
3) then the `p-value`, which is actually the probability of error type 1, that is probability of exceed deviation from E(X) over measured disbalance can be calculated as follows (2 stands for two-sided p-value):

$$
p_{value} = 2 * ( P_{binom}(16,3) + P_{binom}(16,2) + P_{binom}(16,1)) 
$$

In [10]:
from scipy.stats import binom

In [11]:
p_val = 0.0

for k in [1,2,3]:
    p_val += binom.pmf(k,16,0.5)

p_val *= 2.0

print('\np-value =', round(p_val,4))


p-value = 0.0212


# Estimation of sampling length

* In fact, the proposed solution now does not depend on wideness of the histogram - only upon disbalance around 0.5
* more funny that even `p-value` does not show dependence, yet it is evident, that probability of error of type 1 should be also higher if median for histogram is far from 0.5, rather than if it were very close
* let's try to estimate `n_sam` - length of sampling for each of 16 observations 